# Project Phase 2 — Statistical Damage Detection

**Course:** CIV E 789 Sensing Techniques and Data Analytics (Fall 2026)  
**Instructor:** Mohammad Talebi-Kalaleh, Ph.D.  
**Group:** 6 (sensors S2, S4, S6, S9, S12, S15, S18, S21, S24)  
**Data:** `phase2_modal_parameters.json` (intact, damage1, damage2; 20 estimates each)

## Contents

0. [Setup](#setup)
1. [Part A: Feature Distribution Analysis](#part-a)
2. [Part B: Single Feature Damage Detection](#part-b)
3. [Part C: Bivariate Analysis](#part-c)
4. [Part D: Feature Fusion and Engineering](#part-d)
5. [Part E: Principal Component Analysis](#part-e)
6. [Part F: Engineering Insights](#part-f)
7. [Use of Generative AI](#ai-note)

<a id="setup"></a>
## 0. Setup

En esta sección se cargan los datos que usa todo el proyecto y se dejan listas las herramientas comunes. Las Partes A–E parten de los arreglos que se crean aquí, para que todas trabajen con exactamente los mismos datos.

- **0.1** Librerías, rutas y funciones para presentar tablas y figuras.
- **0.2** Carga del JSON y sensores del grupo 6.
- **0.3** Funciones del código base del enunciado.
- **0.4** Extracción de frecuencias naturales y formas modales, con revisiones automáticas.
- **0.5** Resumen de los datos extraídos.

### 0.1 Librerías, rutas y funciones de presentación

- El archivo `phase2_modal_parameters.json` se busca primero en la carpeta del notebook y, si no está, en la carpeta de arriba.
- Las figuras se guardan en `figures/part-a/` … `figures/part-e/` y las tablas de resultados en `results/` (CSV). Las carpetas se crean si no existen.
- `show_table` pone el título de la tabla arriba (**Tabla N.**) y `show_figure` pone el pie de la figura abajo (**Figura N.**), ambos fuera del objeto. `show_figure` también guarda la figura en la carpeta de su parte.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

np.random.seed(42)  # fixed seed (the analysis itself uses no random numbers)

# Input data: next to this notebook or, if not there, in the parent folder
DATA_FILE = 'phase2_modal_parameters.json'
DATA_PATH = next((p for p in (Path(DATA_FILE), Path('..') / DATA_FILE) if p.exists()), None)
assert DATA_PATH is not None, f'{DATA_FILE} not found next to the notebook or in its parent folder'

# Output folders: one figure folder per part, one folder for result tables
FIG_DIR = Path('figures')
RES_DIR = Path('results')
for part in ['part-a', 'part-b', 'part-c', 'part-d', 'part-e']:
    (FIG_DIR / part).mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(exist_ok=True)


def show_table(df, number, caption, filename=None, precision=4):
    """Show the table caption above the table; optionally save the table as CSV in results/."""
    display(Markdown(f'**Tabla {number}.** {caption}'))
    display(df.style.format(precision=precision))
    if filename is not None:
        df.to_csv(RES_DIR / filename)


def show_figure(fig, number, caption, part, filename):
    """Save the figure in figures/<part>/, show it and put the caption below it."""
    fig.savefig(FIG_DIR / part / filename, dpi=300, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    display(Markdown(f'**Figura {number}.** {caption}'))

### 0.2 Carga de datos y sensores del grupo 6

- El JSON tiene dos bloques: `scenarios` (datos de `intact`, `damage1` y `damage2`) y `metadata` (información de muestreo, sensores de cada grupo y posición de cada sensor).
- Los sensores del grupo se leen de `metadata.sensor_assignments.group_6` y no se escriben a mano, para evitar errores al copiar.
- Cada forma modal es una lista de 24 valores ordenados de S1 a S24, así que el sensor $S_k$ está en el índice $k-1$ de la lista (Tabla 1).
- La posición de cada sensor (`metadata.sensor_positions_m`) servirá como eje x de las formas modales.

In [2]:
with open(DATA_PATH, 'r') as f:
    data = json.load(f)

GROUP = 6
SCENARIOS = ['intact', 'damage1', 'damage2']
MODES = ['mode_1', 'mode_2', 'mode_3']
N_MEAS = data['metadata']['sampling_info']['total_estimates']   # 20 estimates per condition
N_SENSORS_TOTAL = len(data['metadata']['sensor_positions_m'])   # 24 sensors on the bridge

# Sensors assigned to our group, read from the file (not typed by hand)
my_sensors = data['metadata']['sensor_assignments'][f'group_{GROUP}']
SENSOR_IDX = np.array([int(s[1:]) - 1 for s in my_sensors])   # Sk -> list index k - 1
SENSOR_POS = np.array([data['metadata']['sensor_positions_m'][s] for s in my_sensors])   # [m]

sensors_table = pd.DataFrame({'Índice en la lista': SENSOR_IDX, 'Posición (m)': SENSOR_POS},
                             index=pd.Index(my_sensors, name='Sensor'))
show_table(sensors_table, 1, f'Sensores asignados al grupo {GROUP}, su índice en las listas de 24 valores '
           'y su posición a lo largo del puente.', precision=1)

**Tabla 1.** Sensores asignados al grupo 6, su índice en las listas de 24 valores y su posición a lo largo del puente.

,Índice en la lista,Posición (m)
Sensor,,
S2,1,7.5
S4,3,17.5
S6,5,27.5
S9,8,42.5
S12,11,57.5
S15,14,72.5
S18,17,87.5
S21,20,102.5
S24,23,117.5


### 0.3 Funciones del código base

Se copian las funciones del código base del enunciado (sección 7) con los mismos nombres y el mismo comportamiento. Así las celdas de las Partes A y B que se escriban a partir de ese código se pueden pegar aquí sin cambios:

- `extract_frequencies(scenario, mode_key)`: las 20 frecuencias de un modo y un escenario.
- `extract_mode_shape(scenario, measurement_idx, mode_key)`: una forma modal en los 9 sensores del grupo.
- `calculate_MAC(phi_ref, phi_test)`: MAC entre dos formas modales.
- `compute_mean_mode_shape(scenario, mode_key)`: forma modal media de las 20 mediciones.
- `mahalanobis_dist(x, mu, cov)`: distancia de Mahalanobis de un punto.

In [3]:
# Starter code functions (same names and behaviour as in the assignment)

# Extract frequencies for a scenario
def extract_frequencies(scenario, mode_key='mode_1'):
    return np.array(data['scenarios'][scenario]['natural_frequencies'][mode_key])


# Extract mode shape at assigned sensors
def extract_mode_shape(scenario, measurement_idx, mode_key='mode_1'):
    mode_data = data['scenarios'][scenario]['mode_shapes'][mode_key][f'measurement_{measurement_idx}']
    # mode_data is a list of 24 values ordered S1..S24, so S1 -> index 0
    return np.array([mode_data[int(s[1:]) - 1] for s in my_sensors])


# Calculate MAC between two mode shapes
def calculate_MAC(phi_ref, phi_test):
    numerator = np.abs(np.dot(phi_ref, phi_test))**2
    denominator = np.dot(phi_ref, phi_ref) * np.dot(phi_test, phi_test)
    return numerator / denominator


# Compute reference mode shapes (mean of all 20 intact measurements)
def compute_mean_mode_shape(scenario, mode_key):
    """Compute mean mode shape across all 20 measurements"""
    all_shapes = []
    for i in range(1, 21):
        shape = extract_mode_shape(scenario, i, mode_key)
        all_shapes.append(shape)
    return np.mean(all_shapes, axis=0)


# Mahalanobis distance calculation
def mahalanobis_dist(x, mu, cov):
    cov_inv = np.linalg.inv(cov)
    diff = x - mu
    return np.sqrt(diff @ cov_inv @ diff.T)

### 0.4 Extracción de frecuencias y formas modales

Los datos se pasan a arreglos de numpy, uno por escenario:

| Arreglo | Tamaño | Contenido |
|---|---|---|
| `freqs[scenario]` | 20 × 3 | mediciones × modos, frecuencia natural en Hz |
| `shapes[scenario]` | 3 × 20 × 9 | modos × mediciones × sensores del grupo 6 |

- **Supuesto:** el valor $i$ de cada lista de frecuencias y `measurement_i` de las formas modales corresponden al mismo segmento de 30 s. El enunciado no lo dice de forma explícita. Para las Partes A y B no importa, porque analizan cada variable por separado, pero las Partes C, D y E juntan varias variables de una misma medición en un solo punto.
- Las mediciones se leen como `measurement_1` … `measurement_20` en orden numérico, no en el orden de las claves del JSON.
- Las formas modales no se vuelven a normalizar: vienen escaladas a un máximo de 1 en los 24 sensores. El MAC no depende de la escala, así que esto no lo afecta.
- Las revisiones (`assert`) detienen el notebook si falta alguna medición, si una forma modal no tiene 24 valores o si hay valores vacíos (NaN).

In [4]:
# freqs[scenario]:  (N_MEAS, 3) array -> measurements x modes, in Hz
# shapes[scenario]: (3, N_MEAS, 9) array -> modes x measurements x our 9 sensors
# Measurements are read as measurement_1 ... measurement_20 (numeric order), so row i of
# freqs and measurement i of shapes refer to the same 30-s segment.
freqs = {}
shapes = {}
for sc in SCENARIOS:
    freqs[sc] = np.column_stack([extract_frequencies(sc, m) for m in MODES])
    shapes[sc] = np.array([[extract_mode_shape(sc, i, m) for i in range(1, N_MEAS + 1)]
                           for m in MODES])

# Sanity checks on the raw file and on the extracted arrays
for sc in SCENARIOS:
    for m in MODES:
        assert len(data['scenarios'][sc]['natural_frequencies'][m]) == N_MEAS
        mode_shapes = data['scenarios'][sc]['mode_shapes'][m]
        assert set(mode_shapes) == {f'measurement_{i}' for i in range(1, N_MEAS + 1)}
        assert all(len(v) == N_SENSORS_TOTAL for v in mode_shapes.values())
    assert freqs[sc].shape == (N_MEAS, len(MODES))
    assert shapes[sc].shape == (len(MODES), N_MEAS, len(my_sensors))
    assert np.isfinite(freqs[sc]).all() and np.isfinite(shapes[sc]).all()
print('Extraction checks passed: '
      f'{len(SCENARIOS)} scenarios x {len(MODES)} modes x {N_MEAS} measurements x {len(my_sensors)} sensors')

Extraction checks passed: 3 scenarios x 3 modes x 20 measurements x 9 sensors


### 0.5 Resumen de los datos extraídos

La Tabla 2 confirma que hay 20 estimaciones por condición, como pide el enunciado ("Getting started"), y que cada forma modal quedó con los 9 valores del grupo.

In [5]:
summary = pd.DataFrame(
    {'Mediciones': [freqs[sc].shape[0] for sc in SCENARIOS],
     **{f'f{k + 1} media (Hz)': [freqs[sc][:, k].mean() for sc in SCENARIOS] for k in range(len(MODES))},
     'Forma modal: valores por medición': [shapes[sc].shape[2] for sc in SCENARIOS]},
    index=pd.Index(SCENARIOS, name='Escenario'))
show_table(summary, 2, 'Resumen de los datos extraídos por escenario: número de mediciones, frecuencia natural '
           'media de cada modo y número de valores de cada forma modal (los 9 sensores del grupo 6).', precision=3)

**Tabla 2.** Resumen de los datos extraídos por escenario: número de mediciones, frecuencia natural media de cada modo y número de valores de cada forma modal (los 9 sensores del grupo 6).

,Mediciones,f1 media (Hz),f2 media (Hz),f3 media (Hz),Forma modal: valores por medición
Escenario,,,,,
intact,20,1.143,3.014,4.348,9
damage1,20,1.103,2.811,4.191,9
damage2,20,1.093,2.714,4.169,9


<a id="part-a"></a>
## Part A: Feature Distribution Analysis (25 pts)

### A1. Natural Frequency Distributions (8 pts)

### A2. Mode Shape Visualization (9 pts)

### A3. Modal Assurance Criterion (MAC) Distributions (8 pts)

<a id="part-b"></a>
## Part B: Single Feature Damage Detection (20 pts)

### B1. Z-Score Anomaly Detection (10 pts)

### B2. Wasserstein Distance Analysis (7 pts)

### B3. Feature Sensitivity Ranking (3 pts)

<a id="part-c"></a>
## Part C: Bivariate Analysis (20 pts)

### C1. Joint Distribution Visualization (7 pts)

### C2. Mahalanobis Distance Detection (8 pts)

### C3. Bivariate vs Univariate Comparison (5 pts)

<a id="part-d"></a>
## Part D: Feature Fusion and Engineering (15 pts)

### D1. Systematic Combination Testing (10 pts)

### D2. Feature Importance Analysis (5 pts)

<a id="part-e"></a>
## Part E: Principal Component Analysis (15 pts)

### E1. PCA Implementation and Variance Analysis (4 pts)

### E2. First Principal Component Analysis (3 pts)

### E3. Two-Dimensional PCA Space (3 pts)

### E4. Progressive Dimensionality Impact (3 pts)

### E5. Physical Interpretation (2 pts)

<a id="part-f"></a>
## Part F: Engineering Insights (5 pts)

### F1. Method Comparison and Selection (2 pts)

### F2. Feature Selection Strategy (2 pts)

### F3. Practical Implementation Considerations (1 pt)

<a id="ai-note"></a>
## Use of Generative AI